# Power Consumption Prediction - Zone 3
This notebook covers exploratory data analysis, feature engineering (including autoregressive lags and seasonality), and Random Forest modeling to accurately predict Zone 3 power demand.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load Data & Preprocess

In [2]:
df = pd.read_csv('powerconsumption.csv')
df['Datetime'] = pd.to_datetime(df['Datetime'])
df.sort_values('Datetime', inplace=True)
df.head()

## 2. Feature Engineering
We extract time-based features to capture daily/weekly seasonality. Furthermore, because power consumption is highly autocorrelated, we incorporate past demand (lag features at 1 hour and 24 hours prior) to vastly improve short-term forecasting.

In [3]:
df['Month'] = df['Datetime'].dt.month
df['Hour'] = df['Datetime'].dt.hour
df['DayOfWeek'] = df['Datetime'].dt.dayofweek

# 10 min intervals: 6 steps = 1 hr, 144 steps = 24 hrs
df['Lag_1h'] = df['PowerConsumption_Zone3'].shift(6)
df['Lag_24h'] = df['PowerConsumption_Zone3'].shift(144)
df.dropna(inplace=True)

## 3. Train/Test Split (Chronological)

In [4]:
features = ['Temperature', 'Humidity', 'WindSpeed', 'GeneralDiffuseFlows', 'DiffuseFlows', 'Month', 'Hour', 'DayOfWeek', 'Lag_1h', 'Lag_24h']
target = 'PowerConsumption_Zone3'

X = df[features]
y = df[target]

# 80/20 Chronological split
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

## 4. Modeling (Random Forest Regressor)

In [5]:
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

## 5. Evaluation & Performance

In [6]:
y_pred = rf.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")